# Panel mixed logit

Estimate a MixL model that integrates each individual's repeated choices jointly using one draw-specific coefficient vector.

This notebook is self-contained and was executed in the repository's Office
validation environment. Install the released package with
`pip install torchdcm`, then select CPU or CUDA through the `device` argument.


In [1]:
import torch

from IPython.display import HTML, display

from torchdcm import (
    Beta,
    ChoiceDataset,
    MixedLogit,
    RandomCoefficient,
    UtilitySpec,
)
from torchdcm.datasets import make_swissmetro_like

torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"TorchDCM example running on {device}")


TorchDCM example running on cuda


In [2]:
def make_choice_data(n_obs=180, seed=7, *, observation_variables=None):
    frame = make_swissmetro_like(n_obs=n_obs, seed=seed)
    data = ChoiceDataset.from_wide(
        frame,
        alternatives=["TRAIN", "SM", "CAR"],
        choice="choice",
        variables={
            "time": {
                "TRAIN": "time_train",
                "SM": "time_sm",
                "CAR": "time_car",
            },
            "cost": {
                "TRAIN": "cost_train",
                "SM": "cost_sm",
                "CAR": "cost_car",
            },
        },
        obs_variables=observation_variables,
        availability={
            "TRAIN": "avail_train",
            "SM": "avail_sm",
            "CAR": "avail_car",
        },
        individual_id="person_id",
    )
    return frame, data


def make_utility_spec(suffix=""):
    tag = f"_{suffix}" if suffix else ""
    spec = UtilitySpec()
    spec.utility(
        "TRAIN",
        Beta(f"ASC_TRAIN{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "SM",
        Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    spec.utility(
        "CAR",
        Beta(f"ASC_CAR{tag}", init=0.0)
        + Beta(f"B_TIME{tag}", init=-0.01) * "time"
        + Beta(f"B_COST{tag}", init=-0.10) * "cost",
    )
    return spec

frame, data = make_choice_data(n_obs=210, seed=23)
print(
    f"Choice occasions: {data.n_obs}; "
    f"individuals: {data.n_individuals}; "
    f"panel detected: {data.has_panel}"
)
spec = make_utility_spec()


Choice occasions: 210; individuals: 70; panel detected: True


In [3]:
model = MixedLogit(
    spec,
    [RandomCoefficient("B_TIME", sigma_init=0.05)],
    n_draws=24,
    seed=23,
    antithetic=True,
    panel=True,
    device=device,
    max_iter=50,
)
result = model.fit(data)
display(HTML(result.report().to_html()))


Model family,MixedLogit
Estimation objective,Maximum simulated log likelihood
TorchDCM version,0.1.0
PyTorch version,2.12.1+cu130
Python version,3.12.13
Operating system,Linux-6.17.0-35-generic-x86_64-with-glibc2.39
Device,cuda
Tensor dtype,float64
Optimizer,torch.optim.LBFGS
Maximum iterations,50
Line search,strong_wolfe
